In [ ]:
PACKAGES="torch transformers accelerate pandas matplotlib"
!uv pip install {PACKAGES} || pip install {PACKAGES}
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('mps')
print(f"Using device: {device}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("SamLowe/roberta-base-go_emotions")
model = AutoModelForSequenceClassification.from_pretrained("SamLowe/roberta-base-go_emotions", device_map="auto")

train_df = pd.read_csv("data/train.csv")
models = {name: i for i, name in enumerate(sorted(train_df["model"].unique()))}

train_df["model_class"] = train_df["model"].map(models)

# Keep token IDs rectangular and within the model's supported sequence length.
encoded_text = tokenizer(
    train_df["text"].fillna("").tolist(),
    padding=True,
    truncation=True,
    max_length=tokenizer.model_max_length,
    return_tensors="pt",
)ac
train_df["text_tokens"] = encoded_text["input_ids"].tolist()
train_df["attention_mask"] = encoded_text["attention_mask"].tolist()

# model_class is a categorical index, not a numeric regression target.
# model_ohe is its one-hot vector; vector positions match the class indices.
y_train = torch.tensor(train_df["model_class"].to_numpy(), dtype=torch.long)
y_train_ohe = F.one_hot(y_train, num_classes=len(models)).float()
train_df["model_ohe"] = y_train_ohe.tolist()

train_df

In [ ]:
print(train_df["text_tokens"].iloc[0])

X_train = encoded_text["input_ids"]
attention_mask_train = encoded_text["attention_mask"]
# y_train holds class indices; y_train_ohe holds one-hot labels.
train_df[["text_tokens", "model_class", "model_ohe"]]

In [ ]:
class TextClassifierHead(torch.nn.Module):
    def __init__(self, in_features=768, out_features=2, dropout=0.1):
        super().__init__()
        if out_features < 1:
            raise ValueError("out_features must be at least 1")
        self.in_features = in_features
        self.out_features = out_features
        self.dropout = torch.nn.Dropout(dropout)
        self.dense = torch.nn.Linear(in_features, in_features)
        self.out_proj = torch.nn.Linear(in_features, out_features)
        self.softmax = torch.nn.Softmax(dim=-1)

    def forward(self, features, return_probabilities=False, **kwargs):
        # RoBERTa passes hidden states with shape (batch, sequence, hidden).
        x = self.dropout(features[:, 0, :])
        x = torch.tanh(self.dense(x))
        logits = self.out_proj(self.dropout(x))
        # Cross-entropy needs logits; request probabilities for inference.
        return self.softmax(logits) if return_probabilities else logits

    def predict_proba(self, features):
        return self(features, return_probabilities=True)


In [ ]:
# Replace the emotion classifier with a trainable model-label classifier.
old_head_parameter = next(model.classifier.parameters())
dropout = model.config.classifier_dropout
if dropout is None:
    dropout = model.config.hidden_dropout_prob

new_head = TextClassifierHead(
    in_features=model.config.hidden_size,
    out_features=len(models),
    dropout=dropout,
)
new_head.apply(model._init_weights)
model.classifier = new_head.to(
    device=old_head_parameter.device,
    dtype=old_head_parameter.dtype,
)

model.num_labels = len(models)
model.config.num_labels = len(models)
model.config.label2id = dict(models)
model.config.id2label = {index: name for name, index in models.items()}
model.config.problem_type = "single_label_classification"

# Pass integer y_train labels to model(..., labels=...) for cross-entropy loss.
model.classifier


In [ ]:
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset, Subset

seed = 42
batch_size = 8
num_epochs = 3
learning_rate = 2e-5
weight_decay = 0.01
validation_fraction = 0.2
checkpoint_path = Path("checkpoints/best_classifier.pt")

torch.manual_seed(seed)

# Hold out entire prompts to prevent prompt overlap between train and validation.
prompt_ids = train_df["prompt_id"].drop_duplicates().sample(frac=1, random_state=seed)
if len(prompt_ids) < 2:
    raise ValueError("At least two distinct prompts are needed for validation")
num_val_prompts = max(1, min(len(prompt_ids) - 1, round(len(prompt_ids) * validation_fraction)))
is_validation = train_df["prompt_id"].isin(prompt_ids.iloc[:num_val_prompts])
train_indices = torch.tensor((~is_validation).to_numpy()).nonzero(as_tuple=True)[0].tolist()
val_indices = torch.tensor(is_validation.to_numpy()).nonzero(as_tuple=True)[0].tolist()

dataset = TensorDataset(X_train, attention_mask_train, y_train)
train_loader = DataLoader(
    Subset(dataset, train_indices), batch_size=batch_size, shuffle=True,
    generator=torch.Generator().manual_seed(seed),
)
val_loader = DataLoader(Subset(dataset, val_indices), batch_size=batch_size)

# Respect the device selected when the pretrained model was loaded.
device = model.get_input_embeddings().weight.device
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=learning_rate, weight_decay=weight_decay,
)
print(f"Device: {device} | Training: {len(train_indices)} | Validation: {len(val_indices)}")


In [ ]:
from tqdm.auto import tqdm

def run_epoch(model, loader, device, optimizer=None, description="Training"):
    training = optimizer is not None
    model.train(training)
    total_loss, total_correct, total_examples = 0.0, 0, 0

    with torch.set_grad_enabled(training):
        progress = tqdm(loader, desc=description, unit="batch", leave=True)
        for input_ids, attention_mask, labels in progress:
            # Trim trailing padding to the longest text in this batch.
            sequence_length = int(attention_mask.sum(dim=1).max().item())
            input_ids = input_ids[:, :sequence_length].to(device)
            attention_mask = attention_mask[:, :sequence_length].to(device)
            labels = labels.to(device)

            if training:
                optimizer.zero_grad(set_to_none=True)

            outputs = model(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels,
            )
            loss = outputs.loss
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite loss encountered during training/validation")

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            count = labels.size(0)
            total_loss += loss.item() * count
            total_correct += (outputs.logits.argmax(dim=-1) == labels).sum().item()
            total_examples += count
            progress.set_postfix(
                loss=f"{total_loss / total_examples:.4f}",
                accuracy=f"{total_correct / total_examples:.2%}",
                refresh=True,
            )

    return total_loss / total_examples, total_correct / total_examples


history = []
best_val_loss = float("inf")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(1, num_epochs + 1):
    print(f"\nEpoch {epoch}/{num_epochs}", flush=True)
    train_loss, train_accuracy = run_epoch(
        model, train_loader, device, optimizer,
        description=f"Epoch {epoch}/{num_epochs} train",
    )
    val_loss, val_accuracy = run_epoch(
        model, val_loader, device,
        description=f"Epoch {epoch}/{num_epochs} validation",
    )
    history.append({
        "epoch": epoch, "train_loss": train_loss, "train_accuracy": train_accuracy,
        "val_loss": val_loss, "val_accuracy": val_accuracy,
    })
    print(
        f"Epoch {epoch}/{num_epochs} | "
        f"train loss {train_loss:.4f}, accuracy {train_accuracy:.2%} | "
        f"val loss {val_loss:.4f}, accuracy {val_accuracy:.2%}",
        flush=True,
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # Include configuration and label mapping for the custom classifier head.
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": model.config.to_dict(),
            "label2id": dict(models),
            "val_loss": val_loss,
        }, checkpoint_path)
        tokenizer.save_pretrained(checkpoint_path.parent)
        print(f"Saved best checkpoint to {checkpoint_path}")

history_df = pd.DataFrame(history)
history_df


In [ ]:
@torch.no_grad()
def predict_model_probabilities(texts):
    if isinstance(texts, str):
        texts = [texts]
    inputs = tokenizer(
        texts, padding=True, truncation=True,
        max_length=tokenizer.model_max_length, return_tensors="pt",
    ).to(model.get_input_embeddings().weight.device)
    was_training = model.training
    model.eval()
    try:
        probabilities = model.classifier.softmax(model(**inputs).logits)
    finally:
        model.train(was_training)
    # Each column is a model category, ordered by its integer class index.
    class_names = [model.config.id2label[i] for i in range(model.num_labels)]
    return pd.DataFrame(probabilities.cpu().float().numpy(), columns=class_names)

# Example: one probability per model; each row sums to 1.
# predict_model_probabilities(train_df["text"].iloc[:3].tolist())


In [ ]:
import matplotlib.pyplot as plt

# Read the full training dataset independently of the training/validation split.
model_counts = pd.read_csv("data/train.csv")["model"].value_counts()

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(model_counts.index, model_counts.values, color="steelblue")
ax.invert_yaxis()
ax.bar_label(bars, padding=4, fmt="%d")
ax.set_title("Model Frequency in the Training Dataset")
ax.set_xlabel("Number of examples")
ax.set_ylabel("Model")
ax.set_xlim(0, model_counts.max() * 1.12)
ax.xaxis.grid(True, alpha=0.25)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()
